# Google Play Store Analysis
## Oasis Infobyte — Data Analytics Internship | Level 2 | Task 4

**Objective:** Perform a comprehensive analysis of the Google Play Store ecosystem — cleaning messy real-world data, exploring app categories, analysing ratings and pricing trends, and conducting sentiment analysis on user reviews.

**Datasets:**
- `googleplaystore.csv` — App metadata (Kaggle: lava18/google-play-store-apps)
- `googleplaystore_user_reviews.csv` — User reviews with translated text (same Kaggle dataset)

**Tech Stack:** Python · pandas · numpy · matplotlib · seaborn · VADER · Plotly

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Styling
sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.facecolor'] = '#0f172a'
plt.rcParams['axes.facecolor'] = '#1e293b'
plt.rcParams['text.color'] = '#f1f5f9'
plt.rcParams['axes.labelcolor'] = '#cbd5e1'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'

print('All libraries imported successfully.')

## 2. Load Datasets

The two datasets are loaded **separately** as required by the official task specification.

In [ ]:
# ── Dataset 1: Apps ─────────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/lava18/google-play-store-apps
# Place googleplaystore.csv in the same directory as this notebook.

apps_df = pd.read_csv('googleplaystore.csv', on_bad_lines='skip')
print(f'Apps dataset loaded: {apps_df.shape[0]:,} rows × {apps_df.shape[1]} columns')

In [ ]:
# ── Dataset 2: User Reviews ───────────────────────────────────────────────────
# Same Kaggle source — googleplaystore_user_reviews.csv

reviews_df = pd.read_csv('googleplaystore_user_reviews.csv', on_bad_lines='skip')
print(f'Reviews dataset loaded: {reviews_df.shape[0]:,} rows × {reviews_df.shape[1]} columns')

## 3. Inspect Datasets

In [ ]:
print('=== Apps Dataset — First 5 rows ===')
display(apps_df.head())
print('\n=== Apps Dataset — Info ===')
apps_df.info()

In [ ]:
print('=== Reviews Dataset — First 5 rows ===')
display(reviews_df.head())
print('\n=== Reviews Dataset — Info ===')
reviews_df.info()

In [ ]:
print('Apps null counts:')
display(apps_df.isnull().sum().sort_values(ascending=False))
print('\nReviews null counts:')
display(reviews_df.isnull().sum().sort_values(ascending=False))

## 4. Data Cleaning — Apps Dataset

Key issues found in the raw data:
- `Installs` stored as string `'10,000+'` — must strip commas and `+`
- `Size` stored as `'19M'` / `'512k'` — must convert to float (MB)
- `Price` stored as `'$2.99'` — must strip `$` and cast to float
- `Rating` values outside 0–5 must be treated as NaN
- Duplicate rows and duplicate App entries must be resolved

In [ ]:
original_rows = len(apps_df)

# ── 1. Drop exact duplicate rows ────────────────────────────────────────────
apps_df = apps_df.drop_duplicates()
print(f'After dedup: {len(apps_df):,} rows (removed {original_rows - len(apps_df):,})')

# ── 2. Fix Reviews (coerce to int) ──────────────────────────────────────────
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce').fillna(0).astype(int)

# ── 3. Keep one entry per App (highest review count wins) ───────────────────
before_app_dedup = len(apps_df)
apps_df = apps_df.sort_values('Reviews', ascending=False).drop_duplicates(subset='App', keep='first')
print(f'After App dedup: {len(apps_df):,} rows (removed {before_app_dedup - len(apps_df):,})')

# ── 4. Fix Installs ─────────────────────────────────────────────────────────
def parse_installs(val):
    if pd.isna(val): return np.nan
    s = str(val).replace(',', '').replace('+', '').strip()
    if s in ('Free', 'Varies with device', ''): return np.nan
    try: return float(s)
    except: return np.nan

apps_df['Installs'] = apps_df['Installs'].apply(parse_installs)

# ── 5. Fix Size ─────────────────────────────────────────────────────────────
def parse_size(val):
    if pd.isna(val): return np.nan
    s = str(val).strip().upper()
    if s in ('VARIES WITH DEVICE', 'VARIES', ''): return np.nan
    try:
        if s.endswith('M'): return float(s[:-1])
        if s.endswith('K'): return float(s[:-1]) / 1024
        return float(s)
    except: return np.nan

apps_df['Size_MB'] = apps_df['Size'].apply(parse_size)

# ── 6. Fix Price ─────────────────────────────────────────────────────────────
apps_df['Price_USD'] = (
    apps_df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .replace({'Free': '0', 'Everyone': '0', '': '0'})
)
apps_df['Price_USD'] = pd.to_numeric(apps_df['Price_USD'], errors='coerce').fillna(0)

# ── 7. Fix Rating ────────────────────────────────────────────────────────────
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
apps_df.loc[(apps_df['Rating'] < 0) | (apps_df['Rating'] > 5), 'Rating'] = np.nan

# ── 8. Normalize Type ────────────────────────────────────────────────────────
apps_df['Type'] = apps_df['Type'].str.strip().str.capitalize()
apps_df.loc[~apps_df['Type'].isin(['Free', 'Paid']), 'Type'] = np.nan

# ── 9. Drop rows missing App or Category ─────────────────────────────────────
apps_df = apps_df[apps_df['App'].notna() & apps_df['Category'].notna()]
apps_df = apps_df.reset_index(drop=True)

print(f'\nFinal cleaned Apps dataset: {len(apps_df):,} rows')
apps_df[['App', 'Category', 'Rating', 'Reviews', 'Size_MB', 'Installs', 'Type', 'Price_USD']].head()

## 5. Category Analysis

Exploring which categories dominate the Play Store.

In [ ]:
cat_counts = apps_df['Category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

fig, ax = plt.subplots(figsize=(10, 8))
top20 = cat_counts.head(20)
colors = sns.color_palette('viridis', len(top20))
bars = ax.barh(top20['Category'][::-1], top20['Count'][::-1], color=colors[::-1], height=0.72)
ax.bar_label(bars, labels=[f'{v:,}' for v in top20['Count'][::-1]], padding=4, color='#cbd5e1', fontsize=9)
ax.set_xlabel('Number of Apps')
ax.set_title('App Distribution by Category (Top 20)', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Total categories: {cat_counts.shape[0]}')
print(f'Most saturated: {cat_counts.head(5)["Category"].tolist()}')
print(f'Least represented: {cat_counts.tail(5)["Category"].tolist()}')

## 6. Ratings Analysis

In [ ]:
rated = apps_df[apps_df['Rating'].notna()]
print(f'Apps with ratings: {len(rated):,} / {len(apps_df):,}')
print(f'Mean rating: {rated["Rating"].mean():.3f}')
print(f'Median rating: {rated["Rating"].median():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating distribution
axes[0].hist(rated['Rating'], bins=20, color='#6366f1', alpha=0.85, edgecolor='#0f172a')
axes[0].axvline(rated['Rating'].mean(), color='#f97316', linestyle='--', lw=1.5, label=f'Mean: {rated["Rating"].mean():.2f}')
axes[0].axvline(rated['Rating'].median(), color='#22c55e', linestyle='--', lw=1.5, label=f'Median: {rated["Rating"].median():.2f}')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# Average rating by category (top 15)
avg_cat = rated.groupby('Category')['Rating'].mean().sort_values(ascending=False).head(15)
colors2 = sns.color_palette('coolwarm_r', len(avg_cat))
bars2 = axes[1].barh(avg_cat.index[::-1], avg_cat.values[::-1], color=colors2[::-1], height=0.72)
axes[1].bar_label(bars2, labels=[f'{v:.2f}' for v in avg_cat.values[::-1]], padding=3, fontsize=8, color='#cbd5e1')
axes[1].set_xlim(0, 5.2)
axes[1].axvline(rated['Rating'].mean(), color='#f97316', linestyle='--', lw=1.2)
axes[1].set_xlabel('Average Rating')
axes[1].set_title('Avg Rating by Category (Top 15)', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Size vs. Installs Analysis

We use a scatter plot and Pearson correlation to explore the relationship between app size and install count.

In [ ]:
size_inst = apps_df[apps_df['Size_MB'].notna() & apps_df['Installs'].notna()].copy()
size_inst = size_inst[size_inst['Installs'] > 0]

corr = size_inst['Size_MB'].corr(size_inst['Installs'])
print(f'Pearson correlation (Size vs Installs): {corr:.4f}')

sample = size_inst.sample(min(2000, len(size_inst)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(sample['Size_MB'], sample['Installs'], alpha=0.3, s=18, color='#6366f1', edgecolors='none')
ax.set_yscale('log')
ax.set_xlabel('App Size (MB)')
ax.set_ylabel('Installs (log scale)')
ax.set_title(f'App Size vs. Installs  (r = {corr:.3f})', fontweight='bold', fontsize=12, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}' if v >= 1 else f'{v:.1f}'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

strength = 'very weak' if abs(corr) < 0.2 else 'weak' if abs(corr) < 0.4 else 'moderate' if abs(corr) < 0.6 else 'strong'
print(f'\nInterpretation: {strength} {"positive" if corr >= 0 else "negative"} relationship.')
print('Note: Correlation does not imply causation.')

## 8. Pricing Analysis

In [ ]:
type_counts = apps_df['Type'].value_counts()
free_count = type_counts.get('Free', 0)
paid_count = type_counts.get('Paid', 0)
total = free_count + paid_count
print(f'Free: {free_count:,} ({free_count/total*100:.1f}%)  |  Paid: {paid_count:,} ({paid_count/total*100:.1f}%)')

paid_df = apps_df[(apps_df['Type'] == 'Paid') & (apps_df['Price_USD'] > 0)].copy()
print(f'Paid apps with valid price: {len(paid_df):,}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Donut — Free vs Paid
wedges, _, autotexts = axes[0].pie(
    [free_count, paid_count], labels=None,
    colors=['#6366f1','#ec4899'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'width': 0.55, 'edgecolor': '#0f172a', 'linewidth': 2},
    pctdistance=0.75,
)
for at in autotexts: at.set_color('#f1f5f9'); at.set_fontweight('bold'); at.set_fontsize(11)
axes[0].legend(wedges, [f'Free ({free_count:,})', f'Paid ({paid_count:,})'], fontsize=10,
               loc='lower center', bbox_to_anchor=(0.5, -0.08), facecolor='#1e293b', labelcolor='#f1f5f9')
axes[0].set_title('Free vs. Paid Distribution', fontweight='bold')

# Price buckets
bins = [0, 0.99, 2.99, 4.99, 9.99, 19.99, 999]
labels = ['<$1', '$1–$3', '$3–$5', '$5–$10', '$10–$20', '>$20']
paid_df['Bucket'] = pd.cut(paid_df['Price_USD'], bins=bins, labels=labels, right=True)
bucket_counts = paid_df['Bucket'].value_counts().sort_index()
colors3 = sns.color_palette('magma', len(labels))
b = axes[1].bar(labels, [bucket_counts.get(l, 0) for l in labels], color=colors3, width=0.65)
axes[1].bar_label(b, padding=3, color='#cbd5e1', fontsize=9)
axes[1].set_xlabel('Price Range (USD)'); axes[1].set_ylabel('Number of Apps')
axes[1].set_title('Paid App Price Distribution', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 9. Revenue Estimation

> **Disclaimer:** This is an indicative estimate only. `Installs` values in the source data represent the lower bound of reported ranges (e.g. `'10,000+'` → 10,000). Actual installs are likely higher. This does not represent real company revenue.

In [ ]:
rev_df = paid_df[paid_df['Installs'].notna() & paid_df['Category'].notna()].copy()
rev_df['Revenue'] = rev_df['Price_USD'] * rev_df['Installs']
rev_by_cat = rev_df.groupby('Category')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors4 = sns.color_palette('plasma', len(rev_by_cat))
bars4 = ax.barh(rev_by_cat.index[::-1], rev_by_cat.values[::-1], color=colors4[::-1], height=0.72)
ax.bar_label(bars4, labels=['${:,.0f}'.format(v) for v in rev_by_cat.values[::-1]], padding=4, color='#cbd5e1', fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e6:.0f}M' if v >= 1e6 else f'${v/1e3:.0f}K'))
ax.set_xlabel('Estimated Revenue (USD)')
ax.set_title('Estimated Revenue by Category — Paid Apps*', fontweight='bold', fontsize=12, pad=12)
ax.annotate('*Indicative estimate using lower-bound install counts × price.', xy=(0.01, -0.1), xycoords='axes fraction', fontsize=8, color='#94a3b8')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

display(rev_by_cat.apply(lambda x: f'${x:,.0f}').rename('Revenue Estimate'))

## 10. Data Cleaning — User Reviews Dataset

In [ ]:
print(f'Original reviews: {len(reviews_df):,}')

# Drop exact duplicates
reviews_df = reviews_df.drop_duplicates()
print(f'After dedup: {len(reviews_df):,}')

# Drop null/empty review text
reviews_df = reviews_df[reviews_df['Translated_Review'].notna()]
reviews_df = reviews_df[reviews_df['Translated_Review'].str.strip().str.lower() != 'nan']
reviews_df = reviews_df[reviews_df['Translated_Review'].str.strip() != '']

# Drop null App
reviews_df = reviews_df[reviews_df['App'].notna()]
reviews_df = reviews_df.reset_index(drop=True)

print(f'Final cleaned reviews: {len(reviews_df):,}')
display(reviews_df.head())

## 11. Sentiment Analysis using VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based NLP model optimised for short texts and social media. Each review receives a compound score from −1 to +1:
- **Positive**: compound ≥ 0.05
- **Negative**: compound ≤ −0.05
- **Neutral**: −0.05 < compound < 0.05

> The existing `Sentiment` column in the source file is used **only** for validation comparison, not as a replacement for VADER classification.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def classify_vader(text):
    score = analyzer.polarity_scores(str(text))['compound']
    if score >= 0.05:  return ('Positive', round(score, 4))
    elif score <= -0.05: return ('Negative', round(score, 4))
    return ('Neutral', round(score, 4))

results = reviews_df['Translated_Review'].apply(classify_vader)
reviews_df['vader_sentiment'] = results.apply(lambda x: x[0])
reviews_df['vader_compound']  = results.apply(lambda x: x[1])

print('VADER classification complete.')
print(reviews_df['vader_sentiment'].value_counts())

# ── Validation: compare with original Sentiment column ──────────────────────
if 'Sentiment' in reviews_df.columns:
    merged = reviews_df[['vader_sentiment', 'Sentiment']].copy()
    merged['Sentiment'] = merged['Sentiment'].str.strip().str.capitalize()
    merged = merged[merged['Sentiment'].isin(['Positive', 'Negative', 'Neutral'])]
    agreement = (merged['vader_sentiment'] == merged['Sentiment']).mean()
    print(f'\nVADER vs. original label agreement: {agreement*100:.1f}%')
    print('(Original labels used for validation only — VADER results drive this analysis)')

In [ ]:
# Distribution visualization
dist = reviews_df['vader_sentiment'].value_counts()
total_reviews = len(reviews_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Donut
labels_s = ['Positive', 'Negative', 'Neutral']
counts_s = [dist.get(l, 0) for l in labels_s]
colors_s = ['#22c55e', '#f43f5e', '#94a3b8']
wedges_s, _, ats = axes[0].pie(counts_s, labels=None, colors=colors_s,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'width': 0.55, 'edgecolor': '#0f172a', 'linewidth': 2},
    pctdistance=0.75)
for at in ats: at.set_color('#f1f5f9'); at.set_fontweight('bold'); at.set_fontsize(11)
axes[0].legend(wedges_s, [f'{l} ({c:,})' for l,c in zip(labels_s, counts_s)], fontsize=9,
               loc='lower center', bbox_to_anchor=(0.5, -0.12), facecolor='#1e293b', labelcolor='#f1f5f9')
axes[0].set_title('VADER Sentiment Distribution', fontweight='bold')

# Bar
bv = axes[1].bar(labels_s, counts_s, color=colors_s, width=0.5)
axes[1].bar_label(bv, labels=[f'{c:,}\n({c/total_reviews*100:.1f}%)' for c in counts_s],
                  padding=4, color='#cbd5e1', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_title('Sentiment Counts (VADER)', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 12. Sentiment by Category

The User Reviews dataset does not contain a Category column. We map reviews to categories using the Apps dataset's App → Category lookup.

In [ ]:
# Build App → Category mapping from the cleaned Apps dataset
cat_map = apps_df[['App', 'Category']].drop_duplicates('App').set_index('App')['Category']

reviews_df['Category'] = reviews_df['App'].map(cat_map)
mapped = reviews_df[reviews_df['Category'].notna()]
print(f'Reviews mapped to a category: {len(mapped):,} / {len(reviews_df):,}')

pivot = (
    mapped.groupby(['Category', 'vader_sentiment'])
    .size().unstack(fill_value=0)
    .reset_index()
)
for col in ['Positive', 'Negative', 'Neutral']:
    if col not in pivot.columns: pivot[col] = 0
pivot['Total'] = pivot['Positive'] + pivot['Negative'] + pivot['Neutral']
pivot['PosPct'] = (pivot['Positive'] / pivot['Total'].replace(0, pd.NA) * 100).round(1)
pivot = pivot.sort_values('PosPct', ascending=False)

print(f'\nMost positive category: {pivot.iloc[0]["Category"]} ({pivot.iloc[0]["PosPct"]}% positive)')
print(f'Most negative category: {pivot.sort_values("Negative", ascending=False).iloc[0]["Category"]}')

# Chart
top15 = pivot.head(15)
fig, ax = plt.subplots(figsize=(11, 7))
y = range(len(top15))
ax.barh(list(y), top15['Positive'].tolist(), color='#22c55e', label='Positive', height=0.65)
ax.barh(list(y), top15['Neutral'].tolist(),  left=top15['Positive'].tolist(), color='#94a3b8', label='Neutral', height=0.65)
ax.barh(list(y), top15['Negative'].tolist(),
        left=(top15['Positive'] + top15['Neutral']).tolist(), color='#f43f5e', label='Negative', height=0.65)
ax.set_yticks(list(y))
ax.set_yticklabels(top15['Category'].tolist(), fontsize=9)
ax.set_xlabel('Number of Reviews')
ax.set_title('Sentiment by Category (Top 15 by Positive %)', fontweight='bold', fontsize=12)
ax.legend(fontsize=9, loc='lower right', facecolor='#1e293b', labelcolor='#f1f5f9')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 13. Plotly Interactive Visualization

In [ ]:
# Interactive: App count by category (top 20) — Plotly bar chart
top20_cat = apps_df['Category'].value_counts().head(20).reset_index()
top20_cat.columns = ['Category', 'Count']

fig_plotly = px.bar(
    top20_cat,
    x='Category', y='Count',
    color='Count',
    color_continuous_scale='Viridis',
    title='Interactive: App Count by Category (Top 20)',
    labels={'Count': 'Number of Apps'},
    template='plotly_dark',
)
fig_plotly.update_layout(
    xaxis_tickangle=-40,
    coloraxis_showscale=False,
    paper_bgcolor='#0f172a',
    plot_bgcolor='#1e293b',
    title_font_size=16,
    margin=dict(t=60, b=120)
)
fig_plotly.show()

In [ ]:
# Interactive sentiment by category (stacked bar)
if len(mapped) > 0:
    fig_sent = go.Figure()
    fig_sent.add_trace(go.Bar(name='Positive', x=top15['Category'].tolist(), y=top15['Positive'].tolist(), marker_color='#22c55e'))
    fig_sent.add_trace(go.Bar(name='Neutral',  x=top15['Category'].tolist(), y=top15['Neutral'].tolist(),  marker_color='#94a3b8'))
    fig_sent.add_trace(go.Bar(name='Negative', x=top15['Category'].tolist(), y=top15['Negative'].tolist(), marker_color='#f43f5e'))
    fig_sent.update_layout(
        barmode='stack',
        title='Interactive: Sentiment by Category (Top 15)',
        xaxis_tickangle=-40,
        template='plotly_dark',
        paper_bgcolor='#0f172a',
        plot_bgcolor='#1e293b',
        margin=dict(t=60, b=120)
    )
    fig_sent.show()

## 14. Developer Insights

Data-driven recommendations for a developer planning to launch a new Android app.

In [ ]:
# ── Insight 1: Highest-rated category (≥ 50 apps) ──────────────────────────
cat_stats = apps_df[apps_df['Rating'].notna()].groupby('Category').agg(
    count=('App','count'), avg_rating=('Rating','mean')
).reset_index()
top_cat = cat_stats[cat_stats['count'] >= 50].sort_values('avg_rating', ascending=False).iloc[0]
print(f'1. Highest-rated category: {top_cat["Category"]} (avg {top_cat["avg_rating"]:.2f})')

# ── Insight 2: Free vs Paid install gap ─────────────────────────────────────
inst = apps_df[apps_df['Installs'].notna() & apps_df['Type'].notna()]
med = inst.groupby('Type')['Installs'].median()
if 'Free' in med and 'Paid' in med:
    print(f'2. Median installs — Free: {int(med["Free"]):,}  |  Paid: {int(med["Paid"]):,}  (ratio: {med["Free"]/max(med["Paid"],1):.0f}×)')

# ── Insight 3: Optimal size bucket ──────────────────────────────────────────
sub = apps_df[apps_df['Size_MB'].notna() & apps_df['Installs'].notna() & (apps_df['Installs'] > 0)].copy()
sub['SizeBucket'] = pd.cut(sub['Size_MB'], bins=[0,10,25,50,100,999],
                            labels=['<10 MB','10–25 MB','25–50 MB','50–100 MB','>100 MB'])
best = sub.groupby('SizeBucket', observed=True)['Installs'].median().idxmax()
print(f'3. Optimal app size for installs: {best}')

# ── Insight 4: Opportunity gap ──────────────────────────────────────────────
opp = pd.DataFrame({'count': cat_counts.set_index('Category')['Count'],
                    'avg_rating': avg_cat}).dropna()
opp['score'] = opp['avg_rating'] / opp['count']
print(f'4. Opportunity gap category: {opp.sort_values("score", ascending=False).index[0]}')

# ── Insight 5: Review volume vs ratings ─────────────────────────────────────
has_rev = apps_df[apps_df['Reviews'].notna() & apps_df['Rating'].notna() & (apps_df['Reviews'] > 0)]
top25 = has_rev[has_rev['Reviews'] > has_rev['Reviews'].quantile(0.75)]['Rating'].mean()
bot25 = has_rev[has_rev['Reviews'] <= has_rev['Reviews'].quantile(0.25)]['Rating'].mean()
print(f'5. Avg rating — top 25%% by reviews: {top25:.2f}  |  bottom 25%%: {bot25:.2f}')

## 15. Conclusion

### Key Findings

1. **Category saturation:** Family, Game, and Tools are the most crowded categories. New entrants face stiff competition.
2. **Free apps dominate installs:** Median installs for free apps far exceed paid apps. A freemium or ad-supported monetisation model maximises reach.
3. **Rating concentration at 4–5:** The vast majority of rated apps sit between 4.0 and 5.0, suggesting either genuine quality or review bias. Apps below 3.5 risk poor discoverability.
4. **Size has a weak correlation with installs:** The Pearson r is low, indicating that a lean app is not inherently disadvantaged — what matters more is category and quality.
5. **Positive sentiment prevails:** VADER analysis shows the majority of user reviews are positive, though negative reviews cluster around specific frustration themes (crashes, ads, updates).
6. **Opportunity categories exist:** Some categories have high average ratings but few apps — these are potential niches where quality products can gain outsized visibility.

### For Developers
- Launch in a high-rating, lower-competition niche.
- Choose a freemium model to maximise install volume.
- Keep app size lean (under 25 MB) for the broadest device compatibility.
- Invest in user experience to generate positive reviews — the data shows a strong positive-sentiment loop among highly reviewed apps.